<a href="https://colab.research.google.com/github/sayeru-dev/Group-Agentic-AI-in-Supply-Chain-BUS4-118S-/blob/main/Agentic_AI_in_Supply_Chain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip -q install pandas

from collections import defaultdict
from math import ceil
from pathlib import Path
import pandas as pd
import numpy as np

In [11]:
# Create 60 days of mock sales data for two SKUs
dates = pd.date_range("2026-01-01", periods=60, freq="D")

sales_rows = []

for i, date in enumerate(dates):
    demand_a = 8 + ((i * 3) % 5) + (5 if i % 11 == 0 else 0)
    demand_b = 5 + ((i * 2) % 4) + (4 if i % 13 == 0 else 0)

    sales_rows.append({
        "date": date,
        "sku": "A",
        "qty_sold": demand_a
    })

    sales_rows.append({
        "date": date,
        "sku": "B",
        "qty_sold": demand_b
    })

sales_data = pd.DataFrame(sales_rows)
sales_data.to_csv("sales.csv", index=False)

inventory_data = pd.DataFrame({
    "sku": ["A", "B"],
    "opening_stock": [55, 40]
})

inventory_data.to_csv("inventory.csv", index=False)

params_data = pd.DataFrame({
    "sku": ["A", "B"],
    "unit_cost": [10, 18],
    "holding_cost_per_day": [0.08, 0.12],
    "stockout_cost": [8, 12],
    "lead_time_days": [4, 3],
    "min_order_qty": [10, 8],
    "service_level": [0.95, 0.95]
})

params_data.to_csv("params.csv", index=False)

print("CSV files created successfully.")

CSV files created successfully.


In [12]:
sales = pd.read_csv("sales.csv", parse_dates=["date"])
inventory = pd.read_csv("inventory.csv").set_index("sku")
params = pd.read_csv("params.csv").set_index("sku")

# Create a complete date/SKU grid
all_dates = pd.date_range(
    sales["date"].min(),
    sales["date"].max(),
    freq="D"
)

all_skus = sorted(sales["sku"].unique())

date_sku_grid = pd.MultiIndex.from_product(
    [all_dates, all_skus],
    names=["date", "sku"]
).to_frame(index=False)

# Aggregate demand and fill missing SKU-days with zero
sales_daily = (
    sales.groupby(["date", "sku"], as_index=False)["qty_sold"]
    .sum()
)

sales_daily = (
    date_sku_grid
    .merge(sales_daily, on=["date", "sku"], how="left")
    .fillna({"qty_sold": 0})
    .sort_values(["date", "sku"])
)

print("Sales data:")
display(sales_daily.head())

print("Inventory data:")
display(inventory)

print("Parameter data:")
display(params)

Sales data:


,date,sku,qty_sold
0,2026-01-01,A,13
1,2026-01-01,B,9
2,2026-01-02,A,11
3,2026-01-02,B,7
4,2026-01-03,A,9


Inventory data:


,opening_stock
sku,
A,55
B,40


Parameter data:


,unit_cost,holding_cost_per_day,stockout_cost,lead_time_days,min_order_qty,service_level
sku,,,,,,
A,10,0.08,8,4,10,0.95
B,18,0.12,12,3,8,0.95


In [13]:
ALPHA = 0.30
REVIEW_DAYS = 1
ORDER_COST = 0.0

def get_z_value(service_level):
    """
    Approximate z-score lookup for common service levels.
    """
    lookup = {
        0.90: 1.28,
        0.95: 1.65,
        0.98: 2.05,
        0.99: 2.33
    }

    return lookup.get(round(float(service_level), 2), 1.65)


def calculate_safety_stock(error_history, service_level, lead_time_days):
    """
    Safety stock =
    z-score × forecast-error standard deviation ×
    square root of lead time plus review period.
    """

    if len(error_history) <= 1:
        error_std = 0.0
    else:
        recent_errors = error_history[-30:]
        error_std = pd.Series(recent_errors).std(ddof=0)

    z_value = get_z_value(service_level)

    safety_stock = (
        z_value
        * error_std
        * np.sqrt(lead_time_days + REVIEW_DAYS)
    )

    return safety_stock

In [14]:
def simulate_inventory_agent(
    sales_data,
    inventory_data,
    params_data,
    use_agent=True,
    print_log=True
):
    """
    Simulates inventory day by day.

    If use_agent=True:
        Uses EWMA forecasting, safety stock, lead time,
        minimum order quantity, and reorder logic.

    If use_agent=False:
        Uses a no-order baseline.
    """

    state = {}

    for sku in inventory_data.index:
        state[sku] = {
            "on_hand": float(inventory_data.loc[sku, "opening_stock"]),
            "on_order": 0.0,
            "forecast": None,
            "forecast_errors": [],
            "purchase_orders": defaultdict(float),
            "stockout_days": 0,
            "demand_units": 0.0,
            "served_units": 0.0,
            "holding_cost": 0.0,
            "stockout_cost": 0.0,
            "order_cost": 0.0
        }

    decision_log = []

    simulation_dates = sorted(sales_data["date"].unique())

    for current_date in simulation_dates:

        current_day = sales_data[
            sales_data["date"] == current_date
        ]

        for _, row in current_day.iterrows():

            sku = row["sku"]
            demand = float(row["qty_sold"])

            current_state = state[sku]
            sku_params = params_data.loc[sku]

            # Receive purchase orders arriving today
            arriving_quantity = current_state["purchase_orders"].pop(
                current_date,
                0.0
            )

            current_state["on_hand"] += arriving_quantity
            current_state["on_order"] -= arriving_quantity

            # Initial forecast
            if current_state["forecast"] is None:
                forecast_before_update = max(demand, 1.0)
            else:
                forecast_before_update = current_state["forecast"]

            # Record forecast error
            forecast_error = demand - forecast_before_update
            current_state["forecast_errors"].append(forecast_error)

            # Update EWMA forecast
            updated_forecast = (
                ALPHA * demand
                + (1 - ALPHA) * forecast_before_update
            )

            current_state["forecast"] = updated_forecast

            lead_time = int(sku_params["lead_time_days"])
            minimum_order = float(sku_params["min_order_qty"])
            service_level = float(sku_params["service_level"])

            # Calculate safety stock
            safety_stock = calculate_safety_stock(
                current_state["forecast_errors"],
                service_level,
                lead_time
            )

            review_horizon = lead_time + REVIEW_DAYS

            # Project inventory after expected demand during lead time
            projected_inventory = (
                current_state["on_hand"]
                + current_state["on_order"]
                - updated_forecast * review_horizon
            )

            # Target inventory level
            target_inventory = (
                updated_forecast * review_horizon
                + safety_stock
            )

            order_quantity = 0.0

            # Agent decision
            if use_agent and projected_inventory < safety_stock:

                inventory_position = (
                    current_state["on_hand"]
                    + current_state["on_order"]
                )

                order_quantity = max(
                    minimum_order,
                    ceil(target_inventory - inventory_position)
                )

                # Prevent unrealistic over-ordering
                maximum_order = max(
                    2 * minimum_order,
                    4 * target_inventory
                )

                order_quantity = min(
                    order_quantity,
                    maximum_order
                )

                # Round order up to the minimum order quantity
                order_quantity = (
                    ceil(order_quantity / minimum_order)
                    * minimum_order
                )

                arrival_date = (
                    pd.Timestamp(current_date)
                    + pd.Timedelta(days=lead_time)
                )

                current_state["purchase_orders"][
                    arrival_date
                ] += order_quantity

                current_state["on_order"] += order_quantity
                current_state["order_cost"] += ORDER_COST

                action = "ORDER"

                reason = (
                    f"Projected inventory {projected_inventory:.1f} "
                    f"is below safety stock {safety_stock:.1f}. "
                    f"Lead time is {lead_time} days."
                )

            else:
                action = "WAIT"

                reason = (
                    f"Projected inventory {projected_inventory:.1f} "
                    f"is at or above safety stock {safety_stock:.1f}."
                )

            # Fulfill demand from available inventory
            served_units = min(
                current_state["on_hand"],
                demand
            )

            lost_units = demand - served_units

            current_state["on_hand"] -= served_units

            # Track metrics
            current_state["demand_units"] += demand
            current_state["served_units"] += served_units

            if lost_units > 0:
                current_state["stockout_days"] += 1

            holding_cost = (
                current_state["on_hand"]
                * float(sku_params["holding_cost_per_day"])
            )

            stockout_cost = (
                lost_units
                * float(sku_params["stockout_cost"])
            )

            current_state["holding_cost"] += holding_cost
            current_state["stockout_cost"] += stockout_cost

            decision_log.append({
                "date": pd.Timestamp(current_date).date(),
                "sku": sku,
                "on_hand": round(current_state["on_hand"], 2),
                "on_order": round(current_state["on_order"], 2),
                "forecast": round(updated_forecast, 2),
                "safety_stock": round(safety_stock, 2),
                "projected_inventory": round(projected_inventory, 2),
                "action": action,
                "order_quantity": round(order_quantity, 2),
                "demand": demand,
                "served_units": served_units,
                "lost_units": lost_units,
                "reason": reason
            })

            if print_log and use_agent:
                print(
                    f"{current_date.date()} | "
                    f"SKU {sku} | "
                    f"On hand: {current_state['on_hand']:.0f} | "
                    f"On order: {current_state['on_order']:.0f} | "
                    f"Forecast: {updated_forecast:.2f} | "
                    f"Safety stock: {safety_stock:.2f} | "
                    f"{action} {order_quantity:.0f} | "
                    f"{reason}"
                )

    # Calculate final metrics
    total_demand = sum(
        item["demand_units"]
        for item in state.values()
    )

    total_served = sum(
        item["served_units"]
        for item in state.values()
    )

    total_stockout_days = sum(
        item["stockout_days"]
        for item in state.values()
    )

    total_holding_cost = sum(
        item["holding_cost"]
        for item in state.values()
    )

    total_stockout_cost = sum(
        item["stockout_cost"]
        for item in state.values()
    )

    total_order_cost = sum(
        item["order_cost"]
        for item in state.values()
    )

    total_cost = (
        total_holding_cost
        + total_stockout_cost
        + total_order_cost
    )

    fill_rate = total_served / total_demand

    metrics = {
        "stockout_days": total_stockout_days,
        "fill_rate": fill_rate,
        "holding_cost": total_holding_cost,
        "stockout_cost": total_stockout_cost,
        "order_cost": total_order_cost,
        "total_cost": total_cost
    }

    return metrics, pd.DataFrame(decision_log)

In [15]:
agent_metrics, agent_log = simulate_inventory_agent(
    sales_data=sales_daily,
    inventory_data=inventory,
    params_data=params,
    use_agent=True,
    print_log=True
)

print("\nAgent simulation completed.")

2026-01-01 | SKU A | On hand: 42 | On order: 10 | Forecast: 13.00 | Safety stock: 0.00 | ORDER 10 | Projected inventory -10.0 is below safety stock 0.0. Lead time is 4 days.
2026-01-01 | SKU B | On hand: 31 | On order: 0 | Forecast: 9.00 | Safety stock: 0.00 | WAIT 0 | Projected inventory 4.0 is at or above safety stock 0.0.
2026-01-02 | SKU A | On hand: 31 | On order: 30 | Forecast: 12.40 | Safety stock: 3.69 | ORDER 20 | Projected inventory -10.0 is below safety stock 3.7. Lead time is 4 days.
2026-01-02 | SKU B | On hand: 24 | On order: 8 | Forecast: 8.40 | Safety stock: 3.30 | ORDER 8 | Projected inventory -2.6 is below safety stock 3.3. Lead time is 3 days.
2026-01-03 | SKU A | On hand: 22 | On order: 40 | Forecast: 11.38 | Safety stock: 5.15 | ORDER 10 | Projected inventory 4.1 is below safety stock 5.1. Lead time is 4 days.
2026-01-03 | SKU B | On hand: 19 | On order: 16 | Forecast: 7.38 | Safety stock: 4.60 | ORDER 8 | Projected inventory 2.5 is below safety stock 4.6. Lead tim

In [16]:
baseline_metrics, baseline_log = simulate_inventory_agent(
    sales_data=sales_daily,
    inventory_data=inventory,
    params_data=params,
    use_agent=False,
    print_log=False
)

results = pd.DataFrame([
    {
        "strategy": "EWMA Inventory Agent",
        **agent_metrics
    },
    {
        "strategy": "No-Order Baseline",
        **baseline_metrics
    }
])

results["fill_rate"] = results["fill_rate"].round(3)
results["holding_cost"] = results["holding_cost"].round(2)
results["stockout_cost"] = results["stockout_cost"].round(2)
results["total_cost"] = results["total_cost"].round(2)

display(results)

,strategy,stockout_days,fill_rate,holding_cost,stockout_cost,order_cost,total_cost
0,EWMA Inventory Agent,0,1.000,147.04,0.0,0.0,147.04
1,No-Order Baseline,109,0.094,19.56,8680.0,0.0,8699.56


In [17]:
print("FINAL RESULTS")
print("=" * 50)

print(
    f"Agent stockout days: "
    f"{agent_metrics['stockout_days']}"
)

print(
    f"Agent fill rate: "
    f"{agent_metrics['fill_rate']:.3f}"
)

print(
    f"Agent total cost: "
    f"${agent_metrics['total_cost']:.2f}"
)

print()

print(
    f"Baseline stockout days: "
    f"{baseline_metrics['stockout_days']}"
)

print(
    f"Baseline fill rate: "
    f"{baseline_metrics['fill_rate']:.3f}"
)

print(
    f"Baseline total cost: "
    f"${baseline_metrics['total_cost']:.2f}"
)

print()

print(
    "Trade-off explanation: The inventory agent places orders before "
    "projected inventory reaches the safety-stock threshold. This improves "
    "customer service and reduces stockouts, but it can increase holding "
    "cost because more inventory is carried."
)

FINAL RESULTS
Agent stockout days: 0
Agent fill rate: 1.000
Agent total cost: $147.04

Baseline stockout days: 109
Baseline fill rate: 0.094
Baseline total cost: $8699.56

Trade-off explanation: The inventory agent places orders before projected inventory reaches the safety-stock threshold. This improves customer service and reduces stockouts, but it can increase holding cost because more inventory is carried.


In [18]:
agent_log.to_csv("agent_decision_log.csv", index=False)
results.to_csv("results.csv", index=False)

print("Saved:")
print("- agent_decision_log.csv")
print("- results.csv")
print("- sales.csv")
print("- inventory.csv")
print("- params.csv")

Saved:
- agent_decision_log.csv
- results.csv
- sales.csv
- inventory.csv
- params.csv
